# 03. Differential abundance with MaAsLin2

This is a minimal R template. It only shows how to run MaAsLin2 on prepared input files.

It is not part of the live Python workshop pipeline.

Code for R

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Differential abundance: MaAsLin2
# Input files: tax.csv, path.csv, metadata.csv
# Tasks: K05 vs healthy, K02 vs healthy
# Mixed K02+K05 samples are excluded
# ══════════════════════════════════════════════════════════════════════════════

library(Maaslin2)
library(dplyr)

# ── PATHS ────────────────────────────────────────────────────────────────────

DATA_DIR <- "data"
OUT_DIR  <- "outputs_minimal/maaslin2"

TAX_PATH      <- file.path(DATA_DIR, "tax.csv")
PATHWAY_PATH  <- file.path(DATA_DIR, "path.csv")
METADATA_PATH <- file.path(DATA_DIR, "metadata.csv")

dir.create(OUT_DIR, recursive = TRUE, showWarnings = FALSE)

# ── LOAD INPUT TABLES ────────────────────────────────────────────────────────

tax <- read.csv(TAX_PATH, check.names = FALSE)
pathways <- read.csv(PATHWAY_PATH, check.names = FALSE)
meta <- read.csv(METADATA_PATH, check.names = FALSE)

# Required columns:
# tax.csv      : run + taxonomic features
# path.csv     : run + pathway features
# metadata.csv : run, healthy, K02_caries, K05_gingivitis_periodontitis

# ── PREPARE TAX MATRIX ───────────────────────────────────────────────────────

rownames(tax) <- tax$run
tax$run <- NULL

tax[] <- lapply(tax, as.numeric)
tax[is.na(tax)] <- 0

# ── PREPARE PATHWAY MATRIX ──────────────────────────────────────────────────
# Stratified HUMAnN columns contain "|" and are removed here.
# UNMAPPED and UNINTEGRATED are also removed.

rownames(pathways) <- pathways$run
pathways$run <- NULL

pathways <- pathways[, !colnames(pathways) %in% c("UNMAPPED", "UNINTEGRATED"), drop = FALSE]
pathways <- pathways[, !grepl("\\|", colnames(pathways)), drop = FALSE]

pathways[] <- lapply(pathways, as.numeric)
pathways[is.na(pathways)] <- 0

# ── PREPARE METADATA ─────────────────────────────────────────────────────────

meta$run <- as.character(meta$run)
rownames(meta) <- meta$run

meta$healthy <- as.integer(meta$healthy)
meta$K02_caries <- as.integer(meta$K02_caries)
meta$K05_gingivitis_periodontitis <- as.integer(meta$K05_gingivitis_periodontitis)

mixed_ids <- meta %>%
  filter(K02_caries == 1, K05_gingivitis_periodontitis == 1) %>%
  pull(run)

cat("Mixed K02+K05 excluded:", length(mixed_ids), "\n")


In [ ]:

# ══════════════════════════════════════════════════════════════════════════════
# K05 vs healthy: TAX
# ══════════════════════════════════════════════════════════════════════════════

meta_k05 <- meta %>%
  filter(!run %in% mixed_ids) %>%
  filter(healthy == 1 | K05_gingivitis_periodontitis == 1) %>%
  mutate(target = ifelse(K05_gingivitis_periodontitis == 1, 1, 0)) %>%
  select(run, target)

rownames(meta_k05) <- meta_k05$run

common_ids <- intersect(rownames(tax), rownames(meta_k05))

tax_k05 <- tax[common_ids, , drop = FALSE]
meta_k05 <- meta_k05[common_ids, , drop = FALSE]

cat("\nK05 vs healthy | TAX\n")
cat("Samples:", nrow(meta_k05), "\n")
cat("Healthy:", sum(meta_k05$target == 0), "\n")
cat("K05:", sum(meta_k05$target == 1), "\n")
cat("Features:", ncol(tax_k05), "\n")

Maaslin2(
  input_data     = tax_k05,
  input_metadata = meta_k05,
  output         = file.path(OUT_DIR, "K05_vs_healthy_TAX"),
  fixed_effects  = c("target"),
  normalization  = "NONE",
  transform      = "LOG",
  min_prevalence = 0.0,
  cores          = 4
)

# ══════════════════════════════════════════════════════════════════════════════
# K02 vs healthy: TAX
# ══════════════════════════════════════════════════════════════════════════════

meta_k02 <- meta %>%
  filter(!run %in% mixed_ids) %>%
  filter(healthy == 1 | K02_caries == 1) %>%
  mutate(target = ifelse(K02_caries == 1, 1, 0)) %>%
  select(run, target)

rownames(meta_k02) <- meta_k02$run

common_ids <- intersect(rownames(tax), rownames(meta_k02))

tax_k02 <- tax[common_ids, , drop = FALSE]
meta_k02 <- meta_k02[common_ids, , drop = FALSE]

cat("\nK02 vs healthy | TAX\n")
cat("Samples:", nrow(meta_k02), "\n")
cat("Healthy:", sum(meta_k02$target == 0), "\n")
cat("K02:", sum(meta_k02$target == 1), "\n")
cat("Features:", ncol(tax_k02), "\n")

Maaslin2(
  input_data     = tax_k02,
  input_metadata = meta_k02,
  output         = file.path(OUT_DIR, "K02_vs_healthy_TAX"),
  fixed_effects  = c("target"),
  normalization  = "NONE",
  transform      = "LOG",
  min_prevalence = 0.0,
  cores          = 4
)

# ══════════════════════════════════════════════════════════════════════════════
# K05 vs healthy: PATH
# ══════════════════════════════════════════════════════════════════════════════

common_ids <- intersect(rownames(pathways), rownames(meta_k05))

path_k05 <- pathways[common_ids, , drop = FALSE]
meta_k05_path <- meta_k05[common_ids, , drop = FALSE]

cat("\nK05 vs healthy | PATH\n")
cat("Samples:", nrow(meta_k05_path), "\n")
cat("Healthy:", sum(meta_k05_path$target == 0), "\n")
cat("K05:", sum(meta_k05_path$target == 1), "\n")
cat("Features:", ncol(path_k05), "\n")

Maaslin2(
  input_data     = path_k05,
  input_metadata = meta_k05_path,
  output         = file.path(OUT_DIR, "K05_vs_healthy_PATH"),
  fixed_effects  = c("target"),
  normalization  = "NONE",
  transform      = "LOG",
  min_prevalence = 0.0,
  cores          = 4
)

# ══════════════════════════════════════════════════════════════════════════════
# K02 vs healthy: PATH
# ══════════════════════════════════════════════════════════════════════════════

common_ids <- intersect(rownames(pathways), rownames(meta_k02))

path_k02 <- pathways[common_ids, , drop = FALSE]
meta_k02_path <- meta_k02[common_ids, , drop = FALSE]

cat("\nK02 vs healthy | PATH\n")
cat("Samples:", nrow(meta_k02_path), "\n")
cat("Healthy:", sum(meta_k02_path$target == 0), "\n")
cat("K02:", sum(meta_k02_path$target == 1), "\n")
cat("Features:", ncol(path_k02), "\n")

Maaslin2(
  input_data     = path_k02,
  input_metadata = meta_k02_path,
  output         = file.path(OUT_DIR, "K02_vs_healthy_PATH"),
  fixed_effects  = c("target"),
  normalization  = "NONE",
  transform      = "LOG",
  min_prevalence = 0.0,
  cores          = 4
)

cat("\nDone.\n")



MaAsLin2 writes result tables into the output folders. The main files are usually `all_results.tsv` and `significant_results.tsv`.